<a href="https://colab.research.google.com/github/varunasnv7-cpu/Varun_Info_5731_Spring2026/blob/main/In_Class_Exercise_5%266_Feature_Extraction_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# In-Class Assignment — Python for Feature Extraction
**Time:** 20 minutes  |  **Points:** 20


....


Dataset file: `product_reviews.txt`

## Load the dataset
Download it from **Canvas**, then run the upload cell below to select the file from your computer.


In [1]:
# Upload the dataset file from your computer (Google Colab)
from google.colab import files

uploaded = files.upload()   # choose product_reviews.txt
filename = next(iter(uploaded))
print("Uploaded file name:", filename)


Saving product_reviews.txt to product_reviews (1).txt
Uploaded file name: product_reviews (1).txt


## Q1 (2 point) — Load & Read & inspect

## Note about the header
**Important:** The dataset file includes a **header row** (column names).  
Make sure the header is used as the column names — it **should NOT appear as a data row** in your DataFrame.

Tip: Use the filename variable printed above when reading the file.
After Reading, check that your columns are `id` and `text`.
Print: **(a)** `df.shape`, **(b)** `df.head(3\)`.  


In [11]:
import pandas as pd

# filename (use the same name as uploaded in Colab)
filename = "product_reviews.txt"

# read the file (tab-separated, header included)
df = pd.read_csv(filename, sep="\t")

# check columns
print(df.columns)

# required outputs
print("Shape:", df.shape)
print("\nFirst 3 rows:")
print(df.head(3))

Index(['id', 'text'], dtype='object')
Shape: (10, 2)

First 3 rows:
   id                                               text
0   1  Love this blender! Smoothies are super creamy ...
1   2  Terrible quality... stopped working after 2 da...
2   3      Good value for the price. Shipping was quick.


## Q2 (4 points) — Basic handcrafted features  
Create these columns and then display the DataFrame:
- `word_count` = number of words  
- `char_count` = number of characters  
- `avg_word_len` = average word length (ignore punctuation)  
- `excl_count` = number of `!` characters  

Print: `id, word_count, char_count, avg_word_len, excl_count`.


In [12]:
import pandas as pd
import re

# filename
filename = "product_reviews.txt"

# read data
df = pd.read_csv(filename, sep="\t")

# word count
df["word_count"] = df["text"].apply(lambda x: len(x.split()))

# character count
df["char_count"] = df["text"].apply(len)

# average word length (ignore punctuation)
df["avg_word_len"] = df["text"].apply(
    lambda x: sum(len(re.sub(r'[^\w]', '', word)) for word in x.split()) / len(x.split())
)

# exclamation count
df["excl_count"] = df["text"].apply(lambda x: x.count("!"))

# print required columns
print(df[["id", "word_count", "char_count", "avg_word_len", "excl_count"]])



   id  word_count  char_count  avg_word_len  excl_count
0   1           9          55      5.000000           1
1   2           7          51      5.571429           3
2   3           8          45      4.500000           0
3   4          10          56      4.500000           0
4   5           8          53      5.500000           1
5   6          10          51      4.000000           0
6   7           9          50      4.444444           0
7   8           6          40      5.500000           0
8   9           7          52      6.285714           0
9  10           7          48      5.428571           0


## Q3 (6 points) — Bag-of-Words (CountVectorizer)  
Use `CountVectorizer(stop_words="english")` on `df["text"]`. Print:
1) vocabulary size (number of features)  
2) top 10 words by **total count** across all documents (word + count)


In [13]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

# filename
filename = "product_reviews.txt"

# read data
df = pd.read_csv(filename, sep="\t")

# CountVectorizer with English stopwords
vectorizer = CountVectorizer(stop_words="english")

X = vectorizer.fit_transform(df["text"])

# vocabulary size
vocab_size = len(vectorizer.vocabulary_)
print("Vocabulary size:", vocab_size)

# sum of word counts across all documents
word_counts = X.toarray().sum(axis=0)

# map words to counts
vocab = vectorizer.get_feature_names_out()
word_freq = dict(zip(vocab, word_counts))

# get top 10 words
top_10 = sorted(word_freq.items(), key=lambda x: x[1], reverse=True)[:10]

print("\nTop 10 words by total count:")
for word, count in top_10:
    print(word, count)



Vocabulary size: 50

Top 10 words by total count:
amazing 1
app 1
battery 1
blender 1
box 1
buy 1
charged 1
clear 1
crashing 1
creamy 1


## Q4 (4 points) — Bigram features  
Use `CountVectorizer(stop_words="english", ngram_range=(2,2))`.  
Print the top 5 bigrams by total count (bigram + count).


In [14]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

# filename
filename = "product_reviews.txt"

# read data
df = pd.read_csv(filename, sep="\t")

# Bigram vectorizer
vectorizer = CountVectorizer(stop_words="english", ngram_range=(2,2))

X = vectorizer.fit_transform(df["text"])

# sum counts
bigram_counts = X.toarray().sum(axis=0)

# get bigrams
bigrams = vectorizer.get_feature_names_out()

# map bigrams to counts
bigram_freq = dict(zip(bigrams, bigram_counts))

# top 5 bigrams
top_5 = sorted(bigram_freq.items(), key=lambda x: x[1], reverse=True)[:5]

print("Top 5 bigrams by total count:")
for bg, count in top_5:
    print(bg, count)


Top 5 bigrams by total count:
amazing screen 1
app keeps 1
battery life 1
blender smoothies 1
box damaged 1


## Q5 (4 points) — TF-IDF features  
Use `TfidfVectorizer(stop_words="english", ngram_range=(1,2))`.  
Compute the **average TF-IDF** score of each term across documents and print the top 5 terms (term + avg score).


In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
import numpy as np

# filename
filename = "product_reviews.txt"

# read data
df = pd.read_csv(filename, sep="\t")

# TF-IDF Vectorizer (unigrams + bigrams)
vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1,2))

X = vectorizer.fit_transform(df["text"])

# compute average TF-IDF score for each term
avg_tfidf = np.mean(X.toarray(), axis=0)

# get terms
terms = vectorizer.get_feature_names_out()

# map terms to scores
term_scores = dict(zip(terms, avg_tfidf))

# get top 5 terms
top_5 = sorted(term_scores.items(), key=lambda x: x[1], reverse=True)[:5]

print("Top 5 terms by average TF-IDF score:")
for term, score in top_5:
    print(term, score)



Top 5 terms by average TF-IDF score:
clear 0.03779644730092272
easy 0.03779644730092272
easy instructions 0.03779644730092272
instructions 0.03779644730092272
instructions clear 0.03779644730092272


## Grading Checklist
- Q1: correct prints  
- Q2: correct feature columns + requested display  
- Q3: correct vocabulary size + correct top 10 words by total count  
- Q4: correct top 5 bigrams by total count  
- Q5: correct top 5 TF-IDF terms by average score
